In [8]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

dataset_path = "C:/Users/Yogesh/Desktop/emotional_classification/emotion_project/data/RAVDESS"

In [9]:
def extract_emotion(filename):
    emotion_code = filename.split("-")[2]
    emotion_dict = {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fearful",
        "07": "disgust",
        "08": "surprised"
    }
    return emotion_dict.get(emotion_code)

def extract_mel_spectrogram(file_path):
    audio, sr = librosa.load(file_path, duration=3, offset=0.5)
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    # FIX SHAPE TO (128,128)
    if mel_db.shape[1] < 128:
        pad_width = 128 - mel_db.shape[1]
        mel_db = np.pad(mel_db, ((0,0),(0,pad_width)), mode='constant')
    else:
        mel_db = mel_db[:, :128]
        
    return mel_db

In [10]:
X = []
y = []

count = 0
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(root, file)
            mel = extract_mel_spectrogram(file_path)
            emotion = extract_emotion(file)
            X.append(mel)
            y.append(emotion)
            count += 1
            if count % 200 == 0:
                print("Processed:", count)

X = np.array(X)
y = np.array(y)

print("Data Loaded! Shape:", X.shape)
print("Labels Loaded! Total:", len(y))

Processed: 200
Processed: 400
Processed: 600
Processed: 800
Processed: 1000
Processed: 1200
Processed: 1400
Data Loaded! Shape: (1440, 128, 128)
Labels Loaded! Total: 1440


In [11]:
# Add channel dimension for CNN
X = X[..., np.newaxis]

# Encode labels
encoder = LabelEncoder()
y = encoder.fit_transform(y)
y = to_categorical(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1152, 128, 128, 1)
Test shape: (288, 128, 128, 1)


In [12]:
# Normalize data to 0-1
X_train = X_train / np.max(X_train)
X_test = X_test / np.max(X_test)

print("Normalization Done")

Normalization Done


C:\Users\Yogesh\AppData\Local\Temp\ipykernel_7144\319847106.py:2: RuntimeWarning: divide by zero encountered in divide
  X_train = X_train / np.max(X_train)
C:\Users\Yogesh\AppData\Local\Temp\ipykernel_7144\319847106.py:2: RuntimeWarning: invalid value encountered in divide
  X_train = X_train / np.max(X_train)
C:\Users\Yogesh\AppData\Local\Temp\ipykernel_7144\319847106.py:3: RuntimeWarning: divide by zero encountered in divide
  X_test = X_test / np.max(X_test)
C:\Users\Yogesh\AppData\Local\Temp\ipykernel_7144\319847106.py:3: RuntimeWarning: invalid value encountered in divide
  X_test = X_test / np.max(X_test)


In [13]:
model = Sequential()

# Block 1
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,1)))
model.add(MaxPooling2D((2,2)))

# Block 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

# Block 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

# Flatten
model.add(Flatten())

# Dense layers
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output layer (8 emotions)
model.add(Dense(8, activation='softmax'))

# Compile
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

C:\Users\Yogesh\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,096 (12.61 MB)

 Trainable params: 3,305,096 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 18s 437ms/step - accuracy: 0.1319 - loss: 2.0786 - val_accuracy: 0.1319 - val_loss: 2.0775
Epoch 2/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 14s 386ms/step - accuracy: 0.1337 - loss: 2.0768 - val_accuracy: 0.1319 - val_loss: 2.0757
Epoch 3/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 18s 325ms/step - accuracy: 0.1337 - loss: 2.0752 - val_accuracy: 0.1319 - val_loss: 2.0743
Epoch 4/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 356ms/step - accuracy: 0.1241 - loss: 2.0737 - val_accuracy: 0.1319 - val_loss: 2.0729
Epoch 5/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 355ms/step - accuracy: 0.1337 - loss: 2.0725 - val_accuracy: 0.1319 - val_loss: 2.0716
Epoch 6/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 366ms/step - accuracy: 0.1337 - loss: 2.0712 - val_accuracy: 0.1319 - val_loss: 2.0705
Epoch 7/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 15s 427ms/step - accuracy: 0.1172 - loss: 2.0703 - val_accuracy: 0.1319 - val_loss: 2.0694
Epoch 8/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 354ms/step - accuracy: 0.1337 - loss: 2.0693 - val_accu